# Animate NetCDF Variable Shading

Set the NetCDF file path and variable name below, then run the notebook from top to bottom. The notebook uses the local `modPlotWRF.py` and `modColorCode.py` helpers for WRF metadata, lat/lon indexing, and color tables.

In [1]:
from pathlib import Path
import re
from datetime import datetime

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import BoundaryNorm, ListedColormap
from IPython.display import HTML, display

import modColorCode as color
import modPlotWRF as wrfplot

# Increase the inline animation size limit for notebooks.
plt.rcParams["animation.embed_limit"] = 80


## User settings

In [2]:
# Choose where frames come from.
# "multi_file": one frame per NetCDF file; labels/sort order come from filename time.
# "single_file": all frames come from one NetCDF file's time dimension.
input_mode = "multi_file"

# Single-file mode setting.
infile = "wrfout_d01.nc"  # Used only when input_mode = "single_file".

# Multi-file mode settings.
data_dir = "."
file_pattern = "wrfout_d03_2024-09-06_*"  # Change this glob to your file sequence.
filename_time_regex = r"(\d{4}-\d{2}-\d{2}_\d{2}:\d{2}:\d{2})"
filename_time_format = "%Y-%m-%d_%H:%M:%S"
time_index_in_each_file = 0
cache_frames = True

# Variable to animate.
varname = "T2"  # Change this to the variable you want to shade.

# For 3D variables, this vertical index is selected before plotting.
# For 2D variables, this setting is ignored.
level_index = 0

# Optional lat/lon subset. This uses modPlotWRF.lat_lon_id and WRF XLAT/XLONG.
use_latlon_subset = False
slat, elat = 20.0, 50.0
slon, elon = -100.0, -60.0

# If use_latlon_subset is False, use grid-index subsetting instead.
# Leave any value as None to use the full extent in that direction.
row_start, row_end = None, None
col_start, col_end = None, None

# Color settings. Set clevs to a list for fixed contour levels, or None for auto levels.
# Example: clevs = [250, 260, 270, 280, 290, 300, 310]
clevs = None
cmap_name = "viridis"
auto_level_percentiles = (2, 98)
auto_level_count = 21

# These colors are used when clevs is provided.
min_color = (0.9, 0.9, 1.0)
max_color = (0.0, 0.0, 1.0)
neg_color = (0.0, 0.0, 1.0)
pos_color = (1.0, 0.0, 0.0)

# Animation settings.
interval_ms = 300
save_animation = False
output_file = f"{varname}_animation.gif"


## Load the dataset

In [3]:
def parse_filename_time(path):
    match = re.search(filename_time_regex, path.name)
    if not match:
        return None, path.name

    text = match.group(1)
    if filename_time_format:
        try:
            return datetime.strptime(text, filename_time_format), text
        except ValueError:
            pass
    return None, text


def file_sort_key(path):
    parsed_time, text = parse_filename_time(path)
    if parsed_time is not None:
        return (0, parsed_time, path.name)
    return (1, text, path.name)


def file_time_label(path):
    parsed_time, text = parse_filename_time(path)
    if parsed_time is not None:
        return parsed_time.strftime("%Y-%m-%d %H:%M:%S")
    return text


def open_reference_dataset(path):
    # Try the local WRF loader first. It prints useful WRF grid information.
    # If the file is not a WRF-style file, fall back to xarray directly.
    try:
        ds_ref, dx_ref, nx, ny, nz = wrfplot.loadWRF(str(path))
    except Exception as exc:
        print("modPlotWRF.loadWRF could not infer WRF dimensions; using xarray.open_dataset instead.")
        print(f"Reason: {type(exc).__name__}: {exc}")
        ds_ref = xr.open_dataset(path)
        wrfplot.infoAll(ds_ref)
        dx_ref = getattr(ds_ref, "DX", 1.0)
    return ds_ref, dx_ref


input_mode = input_mode.lower()
if input_mode not in {"single_file", "multi_file"}:
    raise ValueError('input_mode must be either "single_file" or "multi_file"')

if input_mode == "multi_file":
    data_path = Path(data_dir)
    frame_files = sorted([p for p in data_path.glob(file_pattern) if p.is_file()], key=file_sort_key)
    if not frame_files:
        raise FileNotFoundError(f"No files matched {file_pattern!r} in {data_path.resolve()}")
    frame_labels = [file_time_label(path) for path in frame_files]
    infile_path = frame_files[0]
    print(f"Found {len(frame_files)} files for the animation.")
    print(f"First file: {frame_files[0]}")
    print(f"Last file:  {frame_files[-1]}")
else:
    infile_path = Path(infile)
    if not infile_path.exists():
        raise FileNotFoundError(f"Could not find NetCDF file: {infile_path.resolve()}")
    frame_files = [infile_path]
    frame_labels = None

# Use the first file as the reference for dimensions, coordinates, and metadata.
ds, dx_raw = open_reference_dataset(infile_path)

if varname not in ds:
    raise KeyError(f"Variable {varname!r} was not found. Available variables include: {list(ds.data_vars)[:20]}")

da = ds[varname]
dx_raw = float(dx_raw) if np.isscalar(dx_raw) else float(np.asarray(dx_raw).ravel()[0])
dx_km = dx_raw / 1000.0 if dx_raw > 100.0 else dx_raw

print(da)
print(f"Using dx = {dx_km:g} km for grid-index axes when lat/lon coordinates are unavailable.")


Found 24 files for the animation.
First file: wrfout_d03_2024-09-06_00:00:00
Last file:  wrfout_d03_2024-09-06_23:00:00
A-grid dim (dx,nx,ny,nz) for wrfout_d03_2024-09-06_00:00:00 are (np.float32(1000.0), 399, 399, 30)
<xarray.DataArray 'T2' (Time: 1, south_north: 399, west_east: 399)> Size: 637kB
array([[[300.13397, 300.16537, 300.1944 , ..., 302.8856 , 302.8672 ,
         302.8393 ],
        [300.1545 , 300.193  , 300.2281 , ..., 302.91913, 302.86166,
         302.87134],
        [300.17926, 300.22314, 300.2634 , ..., 302.91083, 302.9009 ,
         302.91763],
        ...,
        [300.20044, 300.24152, 300.46304, ..., 303.03027, 303.0462 ,
         303.062  ],
        [300.1905 , 300.22308, 300.24344, ..., 303.02426, 303.0423 ,
         303.05783],
        [300.1907 , 300.19986, 300.18546, ..., 303.01938, 303.0377 ,
         303.05237]]], shape=(1, 399, 399), dtype=float32)
Coordinates:
    XLAT     (Time, south_north, west_east) float32 637kB 20.97 20.97 ... 24.4
    XLONG    (Time

## Helper functions

In [4]:
def guess_time_dim(data_array):
    for name in ("Time", "time", "times", "valid_time"):
        if name in data_array.dims:
            return name
    return None


def guess_xy_dims(data_array):
    candidates = [
        ("south_north", "west_east"),
        ("south_north_stag", "west_east"),
        ("south_north", "west_east_stag"),
        ("lat", "lon"),
        ("latitude", "longitude"),
        ("y", "x"),
    ]
    for ydim, xdim in candidates:
        if ydim in data_array.dims and xdim in data_array.dims:
            return ydim, xdim
    if len(data_array.dims) >= 2:
        return data_array.dims[-2], data_array.dims[-1]
    raise ValueError(f"Could not infer horizontal dimensions from {data_array.dims}")


def guess_level_dim(data_array, time_dim, ydim, xdim):
    vertical_names = (
        "bottom_top", "bottom_top_stag", "level", "lev", "z", "height",
        "pressure", "isobaric", "isobaricInhPa", "altitude",
    )
    for name in vertical_names:
        if name in data_array.dims:
            return name
    used = {dim for dim in (time_dim, ydim, xdim) if dim is not None}
    leftovers = [dim for dim in data_array.dims if dim not in used]
    return leftovers[0] if leftovers else None


def normalize_bounds(start, end, size):
    start = 0 if start is None else int(start)
    end = size if end is None else int(end)
    start = max(0, min(start, size - 1))
    end = max(start + 1, min(end, size))
    return start, end


def drop_time_like(coord):
    out = coord
    for dim in list(out.dims):
        if dim in ("Time", "time", "times", "valid_time"):
            out = out.isel({dim: 0})
    return out


def wrf_time_label(dataset, time_dim, frame_index):
    if "Times" in dataset:
        raw = dataset["Times"].isel({dataset["Times"].dims[0]: frame_index}).values
        flat = np.asarray(raw).ravel()
        if flat.dtype.kind == "S":
            return b"".join(flat).decode("utf-8", errors="ignore").strip()
        return "".join(str(item) for item in flat).strip()
    if time_dim and time_dim in dataset.coords:
        value = dataset[time_dim].values[frame_index]
        return str(value)
    return f"frame {frame_index + 1}"


def frame_time_label(frame_index):
    if input_mode == "multi_file":
        return frame_labels[frame_index]
    return wrf_time_label(ds, time_dim, frame_index)


def level_label(data_array, level_dim, level_index):
    if level_dim is None:
        return ""
    if level_dim in data_array.coords:
        value = data_array[level_dim].values[level_index]
        return f", {level_dim}={value}"
    return f", {level_dim} index={level_index}"


time_dim = guess_time_dim(da)
ydim, xdim = guess_xy_dims(da)
level_dim = guess_level_dim(da, time_dim, ydim, xdim)

print(f"time_dim={time_dim}, level_dim={level_dim}, ydim={ydim}, xdim={xdim}")

if level_dim is not None and not 0 <= level_index < da.sizes[level_dim]:
    raise IndexError(f"level_index={level_index} is outside the range 0:{da.sizes[level_dim] - 1}")


time_dim=Time, level_dim=None, ydim=south_north, xdim=west_east


## Prepare the plot domain and colors

In [5]:
y_size = da.sizes[ydim]
x_size = da.sizes[xdim]

if use_latlon_subset:
    if "XLAT" not in ds or "XLONG" not in ds:
        raise ValueError("use_latlon_subset=True requires WRF XLAT and XLONG variables in the dataset.")
    rs, re, cs, ce = wrfplot.lat_lon_id(ds, slat, elat, slon, elon)
    rs, re = sorted((int(rs), int(re)))
    cs, ce = sorted((int(cs), int(ce)))
    re = min(re + 1, y_size)
    ce = min(ce + 1, x_size)
else:
    rs, re = normalize_bounds(row_start, row_end, y_size)
    cs, ce = normalize_bounds(col_start, col_end, x_size)

row_slice = slice(rs, re)
col_slice = slice(cs, ce)
print(f"Plotting rows {rs}:{re}, columns {cs}:{ce}")

frame_cache = {}


def select_frame_values(dataset, frame_index_in_dataset=0):
    if varname not in dataset:
        raise KeyError(f"Variable {varname!r} was not found in {getattr(dataset, 'encoding', {})}")

    frame = dataset[varname]
    if time_dim is not None and time_dim in frame.dims:
        if not 0 <= frame_index_in_dataset < frame.sizes[time_dim]:
            raise IndexError(
                f"time index {frame_index_in_dataset} is outside the range "
                f"0:{frame.sizes[time_dim] - 1} for {time_dim}"
            )
        frame = frame.isel({time_dim: frame_index_in_dataset})
    if level_dim is not None and level_dim in frame.dims:
        frame = frame.isel({level_dim: level_index})
    frame = frame.squeeze(drop=True)
    if ydim not in frame.dims or xdim not in frame.dims:
        raise ValueError(f"After selecting time/level, expected dims {ydim!r}, {xdim!r}; got {frame.dims}")
    frame = frame.transpose(ydim, xdim)
    values = frame.isel({ydim: row_slice, xdim: col_slice}).values
    return np.asarray(values).copy()


def get_frame(frame_index):
    if cache_frames and frame_index in frame_cache:
        return frame_cache[frame_index]

    if input_mode == "multi_file":
        frame_file = frame_files[frame_index]
        with xr.open_dataset(frame_file) as frame_ds:
            values = select_frame_values(frame_ds, time_index_in_each_file)
    else:
        values = select_frame_values(ds, frame_index)

    if cache_frames:
        frame_cache[frame_index] = values
    return values


def build_xy_grid():
    if "XLAT" in ds and "XLONG" in ds:
        lat = drop_time_like(ds["XLAT"])
        lon = drop_time_like(ds["XLONG"])
        if lat.ndim == 2 and lon.ndim == 2 and lat.shape[0] >= re and lat.shape[1] >= ce:
            return lon.values[row_slice, col_slice], lat.values[row_slice, col_slice], "Longitude", "Latitude"

    if xdim in ds.coords and ydim in ds.coords:
        xvals = np.asarray(ds.coords[xdim].values)[col_slice]
        yvals = np.asarray(ds.coords[ydim].values)[row_slice]
        if xvals.ndim == 1 and yvals.ndim == 1:
            xgrid, ygrid = np.meshgrid(xvals, yvals)
            return xgrid, ygrid, xdim, ydim

    xvals = (np.arange(cs, ce) - 0.5 * (x_size - 1)) * dx_km
    yvals = (np.arange(rs, re) - 0.5 * (y_size - 1)) * dx_km
    xgrid, ygrid = np.meshgrid(xvals, yvals)
    return xgrid, ygrid, "X (km)", "Y (km)"


xgrid, ygrid, xlabel, ylabel = build_xy_grid()
nframes = len(frame_files) if input_mode == "multi_file" else (da.sizes[time_dim] if time_dim is not None else 1)

if clevs is None:
    stride = max(1, nframes // 10)
    samples = []
    for frame_index in range(0, nframes, stride):
        values = np.asarray(get_frame(frame_index), dtype=float)
        samples.append(values[np.isfinite(values)])
    finite_samples = [sample for sample in samples if sample.size]
    if not finite_samples:
        raise ValueError("No finite data values were found in the selected plot domain.")
    samples = np.concatenate(finite_samples)
    vmin, vmax = np.nanpercentile(samples, auto_level_percentiles)
    if np.isclose(vmin, vmax):
        vmin, vmax = float(np.nanmin(samples)), float(np.nanmax(samples))
    if np.isclose(vmin, vmax):
        vmin, vmax = vmin - 0.5, vmax + 0.5
    levels = np.linspace(vmin, vmax, auto_level_count)
    cmap = plt.get_cmap(cmap_name)
    norm = None
else:
    levels = np.asarray(clevs, dtype=float)
    if levels[0] < 0 < levels[-1]:
        cmap, norm = color.make_cmap(levels, neg_color=neg_color, pos_color=pos_color)
    else:
        cmap = ListedColormap(color.myColor(levels, min_color, max_color))
        norm = BoundaryNorm(levels, cmap.N)

print(f"Animating {nframes} frame(s) with levels from {levels[0]:g} to {levels[-1]:g}.")


Plotting rows 0:399, columns 0:399
Animating 24 frame(s) with levels from 295.995 to 303.122.


## Create and display the animation

In [6]:
fig, ax = plt.subplots(figsize=(8, 6))

first_plot = ax.contourf(
    xgrid,
    ygrid,
    get_frame(0),
    levels=levels,
    cmap=cmap,
    norm=norm,
    extend="both",
)

long_name = da.attrs.get("description") or da.attrs.get("long_name") or varname
units = da.attrs.get("units", "")
cbar_label = f"{long_name} ({units})" if units else long_name
cbar = fig.colorbar(first_plot, ax=ax, pad=0.02)
cbar.set_label(cbar_label)

lev_label = level_label(da, level_dim, level_index)


def draw(frame_index):
    ax.clear()
    ax.contourf(
        xgrid,
        ygrid,
        get_frame(frame_index),
        levels=levels,
        cmap=cmap,
        norm=norm,
        extend="both",
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{varname}{lev_label}: {frame_time_label(frame_index)}")
    ax.set_xlim(np.nanmin(xgrid), np.nanmax(xgrid))
    ax.set_ylim(np.nanmin(ygrid), np.nanmax(ygrid))
    return []


ani = FuncAnimation(fig, draw, frames=nframes, interval=interval_ms, blit=False)
plt.close(fig)

display(HTML(ani.to_jshtml()))

if save_animation:
    suffix = Path(output_file).suffix.lower()
    writer = "pillow" if suffix == ".gif" else "ffmpeg"
    ani.save(output_file, writer=writer, dpi=120)
    print(f"Saved animation to {Path(output_file).resolve()}")
